# Phase 1 Validation: Graph and Atlas Ingestion

This notebook is a step-by-step validation of the Phase 1 ingestion workflow.

Goals:

* inspect the structure of `NC_pct21.json`

* inspect the structure of representative Atlas `.jsonl.gz` files

* confirm where district assignments live in the Atlas schema

* show unit ID normalization explicitly

* verify that Atlas plan assignments align exactly with graph unit IDs

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent
SRC_PATH = REPO_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("Repo root:", REPO_ROOT)
print("Src path:", SRC_PATH)

Repo root: /Users/ofosuosei/Desktop/Projects/gerry-dislocation-ensemble
Src path: /Users/ofosuosei/Desktop/Projects/gerry-dislocation-ensemble/src


In [2]:
import gzip
from itertools import islice

from gerry.io.graph import load_graph_json, extract_graph_unit_ids, normalize_unit_id
from gerry.io.atlas import read_atlas_header, iter_plan_records, extract_plan_assignment
from gerry.io.validate import compare_unit_sets

# Paths
GRAPH_PATH = Path("../data/raw/graphs/NC_pct21.json")
NON_COUNTY_ATLAS_PATH = Path(
    "../data/raw/atlas/nc_ush/cyclewalk_gamma0.0/run1.jsonl.gz"
)
COUNTY_ATLAS_PATH = Path("../data/raw/atlas/nc_ush/forestrecom_county/run.jsonl.gz")

In [3]:
##  Quick paths existence check
for path in [GRAPH_PATH, NON_COUNTY_ATLAS_PATH, COUNTY_ATLAS_PATH]:
    print(path, "->", path.exists())

../data/raw/graphs/NC_pct21.json -> True
../data/raw/atlas/nc_ush/cyclewalk_gamma0.0/run1.jsonl.gz -> True
../data/raw/atlas/nc_ush/forestrecom_county/run.jsonl.gz -> True


## Section 1: Inspect the graph file

In [4]:
# Load graph JSON
graph_data = load_graph_json(GRAPH_PATH)
type(graph_data)

dict

In [5]:
# Top-level graph structure
list(graph_data.keys())[:20]

['directed', 'multigraph', 'graph', 'nodes', 'adjacency']

In [6]:
# Inspect raw node container
nodes = graph_data.get("nodes")
type(nodes)

list

In [7]:
# Preview node contents
if isinstance(nodes, dict):
    print("Number of node entries:", len(nodes))
    for i, (k, v) in enumerate(nodes.items()):
        print("NODE KEY:", k)
        print("NODE VALUE:", v)
        print("-" * 60)
        if i == 2:
            break
elif isinstance(nodes, list):
    print("Number of node entries:", len(nodes))
    for item in nodes[:1]:
        print(item)
        print("-" * 60)
else:
    print("Unexpected node structure:", type(nodes))

Number of node entries: 2650
{'id': 0, 'county': 'ALAMANCE', 'prec_id': '01', 'pop2020cen': 5135, 'G20_AG_D': 618.0000000000002, 'G20_AG_R': 2266.0000000000005, 'G20_AD_D': 608.0, 'G20_AD_R': 2246.999999999999, 'G20_CA_D': 485.0, 'G20_CA_R': 2400.999999999998, 'G20_CI_D': 561.0, 'G20_CI_R': 2303.9999999999995, 'G20_CL_D': 572.9999999999999, 'G20_CL_R': 2293.000000000001, 'G20_GV_D': 661.0000000000003, 'G20_GV_R': 2209.0000000000005, 'G20_LG_D': 556.0000000000002, 'G20_LG_R': 2336.0000000000014, 'G20_SST_D': 617.0000000000001, 'G20_SST_R': 2250.9999999999986, 'G20_TR_D': 611.0000000000003, 'G20_TR_R': 2240.0000000000005, 'G20_PR_D': 566.0, 'G20_PR_R': 2299.0000000000005, 'G20_USS_D': 564.0000000000001, 'G20_USS_R': 2205.0, 'G08_PR_D': 565.0, 'G08_PR_R': 1423.9999999999989, 'G08_AG_D': 833.9999999999999, 'G08_AG_R': 1134.0, 'G08_AD_D': 669.9999999999994, 'G08_AD_R': 1234.0000000000002, 'G08_CA_D': 448.00000000000006, 'G08_CA_R': 1521.0000000000011, 'G08_CI_D': 587.9999999999999, 'G08_CI_

In [8]:
# Extract graph unit IDs
graph_unit_ids = extract_graph_unit_ids(graph_data)
len(graph_unit_ids), graph_unit_ids[:10]

(2650,
 ['ALAMANCE_01',
  'ALAMANCE_02',
  'ALAMANCE_035',
  'ALAMANCE_03C',
  'ALAMANCE_03N',
  'ALAMANCE_03N2',
  'ALAMANCE_03S',
  'ALAMANCE_03W',
  'ALAMANCE_04',
  'ALAMANCE_05'])

In [9]:
# Confirm uniqueness
len(graph_unit_ids), len(set(graph_unit_ids)), len(graph_unit_ids) == len(set(graph_unit_ids))

(2650, 2650, True)

In [10]:
# Expected graph count check
assert len(graph_unit_ids) == 2650
print("PASS: graph has 2,650 unit IDs")

PASS: graph has 2,650 unit IDs


## Section 2: Inspect the raw Atlas file

In [11]:
#  Show first few raw lines from non-county atlas

with gzip.open(NON_COUNTY_ATLAS_PATH, "rt", encoding="utf-8") as f:
    for i, line in enumerate(islice(f, 5), start=1):
        print(f"LINE {i}:")
        print(line[:300])
        print("=" * 80)

LINE 1:
"This is an Atlas for Redistricting Maps"

LINE 2:
{"description":"","date":"2025-04-21T13:00:27.822","atlasParamType":"Dict{String, Any}","mapParamType":"Dict{String, Any}"}

LINE 3:
{"energy weights":[],"population bounds":[730758,760583],"commit":"f167343cfb3c903f9e640ce6a100a361d7abdb78","energies":[],"districts":14}

LINE 4:
{"name":"step1","weight":1,"data":{"get_log_spanning_forests":3542.7431395137996,"get_log_spanning_trees":[163.5840750050144,277.2716354679154,217.29747633144908,271.48344976837535,317.8287087860894,180.38314199012294,273.0114787112698,403.4549113882553,279.15122566192815,405.77210002776104,187.7385
LINE 5:
{"name":"step100000","weight":1,"data":{"get_log_spanning_forests":3476.555415578629,"get_log_spanning_trees":[247.3693460046862,274.97693421812926,380.566055247597,182.34554650461968,183.52260384643338,417.115532942913,221.6436495891805,138.5891170524478,267.25010524362307,280.89667202302456,302.35


In [12]:
#  Read and inspect Atlas header
header = read_atlas_header(NON_COUNTY_ATLAS_PATH)
len(header)

3

In [13]:
# Inspect header record types
[(i + 1, type(record).__name__) for i, record in enumerate(header)]

[(1, 'str'), (2, 'dict'), (3, 'dict')]

In [14]:
# Preview header records
for i, record in enumerate(header, start=1):
    print(f"HEADER RECORD {i}")
    print(record if isinstance(record, str) else str(record)[:500])
    print("=" * 80)

HEADER RECORD 1
This is an Atlas for Redistricting Maps
HEADER RECORD 2
{'description': '', 'date': '2025-04-21T13:00:27.822', 'atlasParamType': 'Dict{String, Any}', 'mapParamType': 'Dict{String, Any}'}
HEADER RECORD 3
{'energy weights': [], 'population bounds': [730758, 760583], 'commit': 'f167343cfb3c903f9e640ce6a100a361d7abdb78', 'energies': [], 'districts': 14}


## Section 3: Inspect the first plan record

In [15]:
#  Load first plan record
first_plan = next(iter_plan_records(NON_COUNTY_ATLAS_PATH))
type(first_plan)

dict

In [16]:
# Inspect plan record keys
list(first_plan.keys())

['name', 'weight', 'data', 'districting']

In [17]:
# Inspect key fields
for key in ["name", "weight", "data", "districting"]:
    value = first_plan.get(key)
    print(f"\nFIELD: {key}")
    print("TYPE:", type(value).__name__)

    if isinstance(value, dict):
        print("N ITEMS:", len(value))
        for i, (k, v) in enumerate(value.items()):
            print(" ", k, "->", v)
            if i == 4:
                break

    elif isinstance(value, list):
        print("LENGTH:", len(value))
        for i, item in enumerate(value[:5]):
            print(" ", item)

    else:
        print(str(value)[:500])


FIELD: name
TYPE: str
step1

FIELD: weight
TYPE: int
1

FIELD: data
TYPE: dict
N ITEMS: 3
  get_log_spanning_forests -> 3542.7431395137996
  get_log_spanning_trees -> [163.5840750050144, 277.2716354679154, 217.29747633144908, 271.48344976837535, 317.8287087860894, 180.38314199012294, 273.0114787112698, 403.4549113882553, 279.15122566192815, 405.77210002776104, 187.7385138239343, 198.80962606250858, 179.08336154399365, 187.87343494518248]
  get_isoperimetric_scores -> [81.36787805899989, 116.26044801239182, 42.82338911509231, 50.05648218637503, 66.23284418400766, 70.02142772278931, 72.42822454154208, 47.87726268087349, 52.69124562735151, 35.73394170098486, 96.05083317986416, 79.32137712578918, 94.70069374407034, 55.07758661500155]

FIELD: districting
TYPE: list
LENGTH: 2650
  {'["DAVIDSON_18"]': 2}
  {'["JOHNSTON_PR28"]': 13}
  {'["MCDOWELL_T.COVE"]': 8}
  {'["CUMBERLAND_CC14"]': 12}
  {'["MARTIN_BG"]': 10}


## Section 4: Inspect normalization

In [18]:
# Show raw districting field
districting = first_plan["districting"]
type(districting), len(districting)
districting[:10]

[{'["DAVIDSON_18"]': 2},
 {'["JOHNSTON_PR28"]': 13},
 {'["MCDOWELL_T.COVE"]': 8},
 {'["CUMBERLAND_CC14"]': 12},
 {'["MARTIN_BG"]': 10},
 {'["ORANGE_WD"]': 3},
 {'["WAYNE_04"]': 13},
 {'["CATAWBA_31"]': 6},
 {'["DAVIE_01"]': 6},
 {'["CATAWBA_20"]': 4}]

In [19]:
# Show raw atlas keys
raw_districting_keys = [list(item.keys())[0] for item in districting[:10]]
raw_districting_keys

['["DAVIDSON_18"]',
 '["JOHNSTON_PR28"]',
 '["MCDOWELL_T.COVE"]',
 '["CUMBERLAND_CC14"]',
 '["MARTIN_BG"]',
 '["ORANGE_WD"]',
 '["WAYNE_04"]',
 '["CATAWBA_31"]',
 '["DAVIE_01"]',
 '["CATAWBA_20"]']

In [20]:
# Show normalized keys
[(k, normalize_unit_id(k)) for k in raw_districting_keys]

[('["DAVIDSON_18"]', '["DAVIDSON_18"]'),
 ('["JOHNSTON_PR28"]', '["JOHNSTON_PR28"]'),
 ('["MCDOWELL_T.COVE"]', '["MCDOWELL_T.COVE"]'),
 ('["CUMBERLAND_CC14"]', '["CUMBERLAND_CC14"]'),
 ('["MARTIN_BG"]', '["MARTIN_BG"]'),
 ('["ORANGE_WD"]', '["ORANGE_WD"]'),
 ('["WAYNE_04"]', '["WAYNE_04"]'),
 ('["CATAWBA_31"]', '["CATAWBA_31"]'),
 ('["DAVIE_01"]', '["DAVIE_01"]'),
 ('["CATAWBA_20"]', '["CATAWBA_20"]')]

### Normalization note

* At this stage, normalization is only being used to convert Atlas unit keys into a comparable graph-unit format.
* Raw atlas keys are wrapped string representations; the final cleaned unit IDs are produced during assignment extraction.

## Section 5: Extract assignment and inspect it

In [21]:
# Section 5: Extract assignment and inspect it
assignment = extract_plan_assignment(first_plan)
len(assignment)

2650

In [22]:
# Show sample unit -> district pairs
list(assignment.items())[:10]

[('DAVIDSON_18', 2),
 ('JOHNSTON_PR28', 13),
 ('MCDOWELL_T.COVE', 8),
 ('CUMBERLAND_CC14', 12),
 ('MARTIN_BG', 10),
 ('ORANGE_WD', 3),
 ('WAYNE_04', 13),
 ('CATAWBA_31', 6),
 ('DAVIE_01', 6),
 ('CATAWBA_20', 4)]

In [23]:
# Inspect unique district labels
unique_districts = sorted({str(v) for v in assignment.values()})
len(unique_districts), unique_districts[:20]

(14,
 ['1', '10', '11', '12', '13', '14', '2', '3', '4', '5', '6', '7', '8', '9'])

## Section 6: Compare atlas IDs to graph IDs

In [24]:
# Compare unit sets and print comparison summary
comparison = compare_unit_sets(list(assignment.keys()), graph_unit_ids)

print("Graph unit count:", comparison["graph_count"])
print("Assignment unit count:", comparison["assignment_count"])
print("Missing from assignment:", len(comparison["missing_from_assignment"]))
print("Extra in assignment:", len(comparison["extra_in_assignment"]))

if comparison["missing_from_assignment"]:
    print("\nSample missing IDs:")
    print(comparison["missing_from_assignment"][:10])

if comparison["extra_in_assignment"]:
    print("\nSample extra IDs:")
    print(comparison["extra_in_assignment"][:10])

Graph unit count: 2650
Assignment unit count: 2650
Missing from assignment: 0
Extra in assignment: 0


In [25]:
# Exact match assertion
assert comparison["graph_count"] == 2650
assert comparison["assignment_count"] == 2650
assert len(comparison["missing_from_assignment"]) == 0
assert len(comparison["extra_in_assignment"]) == 0
print("PASS: first Atlas plan assignment matches graph unit IDs exactly")

PASS: first Atlas plan assignment matches graph unit IDs exactly


## Section 7: Repeat on county-preserving atlas

In [26]:
# Inspect county-preserving header
county_header = read_atlas_header(COUNTY_ATLAS_PATH)
[(i + 1, type(record).__name__) for i, record in enumerate(county_header)]

[(1, 'str'), (2, 'dict'), (3, 'dict')]

In [27]:
# Load first county-preserving plan
county_first_plan = next(iter_plan_records(COUNTY_ATLAS_PATH))
list(county_first_plan.keys())

['name', 'weight', 'data', 'districting']

In [28]:
# Extract county assignment
county_assignment = extract_plan_assignment(county_first_plan)
len(county_assignment), list(county_assignment.items())[:10]

(700,
 [('["MECKLENBURG", "052"]', 13),
  ('["MONTGOMERY", "CHE"]', 11),
  ('GASTON', 14),
  ('HENDERSON', 7),
  ('["WAKE", "20-09"]', 2),
  ('["ORANGE", "OW1"]', 11),
  ('["CLEVELAND", "KM N"]', 14),
  ('["CABARRUS", "02-01"]', 12),
  ('["WAKE", "12-04"]', 2),
  ('["ORANGE", "CH"]', 11)])

In [29]:
county_comparison = compare_unit_sets(list(county_assignment.keys()), graph_unit_ids)

print("Graph unit count:", county_comparison["graph_count"])
print("Assignment unit count:", county_comparison["assignment_count"])
print("Missing from assignment:", len(county_comparison["missing_from_assignment"]))
print("Extra in assignment:", len(county_comparison["extra_in_assignment"]))

print("\nSample county assignment keys:")
print(list(county_assignment.keys())[:10])

print("\nObservation:")
if county_comparison["assignment_count"] != county_comparison["graph_count"]:
    print(
        "County-preserving atlas is using a different unit system from NC_pct21.json. "
    )
else:
    print("County-preserving atlas appears to use the same unit system as NC_pct21.json.")

Graph unit count: 2650
Assignment unit count: 700
Missing from assignment: 2650
Extra in assignment: 700

Sample county assignment keys:
['["MECKLENBURG", "052"]', '["MONTGOMERY", "CHE"]', 'GASTON', 'HENDERSON', '["WAKE", "20-09"]', '["ORANGE", "OW1"]', '["CLEVELAND", "KM N"]', '["CABARRUS", "02-01"]', '["WAKE", "12-04"]', '["ORANGE", "CH"]']

Observation:
County-preserving atlas is using a different unit system from NC_pct21.json. 


In [30]:
# Inspect the county atlas header
for i, record in enumerate(county_header, start=1):
    print(f"\nHEADER RECORD {i}")
    print(record)
    print("=" * 100)


HEADER RECORD 1
This is an Atlas for Redistricting Maps

HEADER RECORD 2
{'description': '', 'date': '2021-11-04T14:11:31.414', 'atlasParamType': 'Dict{String, Any}', 'mapParamType': 'Dict{String, Any}'}

HEADER RECORD 3
{'energy weights': [0.05], 'levels in graph': ['county', 'prec_id'], 'population bounds': [738214, 753127], 'graph nodes by level': [100, 2650], 'gamma': 0.0, 'graph edges by level': [244, 7490], 'energies': ['get_isoperimetric_score'], 'districts': 14}


In [31]:
county_districting = county_first_plan["districting"]
county_districting[:20]

[{'["MECKLENBURG", "052"]': 13},
 {'["MONTGOMERY", "CHE"]': 11},
 {'["GASTON"]': 14},
 {'["HENDERSON"]': 7},
 {'["WAKE", "20-09"]': 2},
 {'["ORANGE", "OW1"]': 11},
 {'["CLEVELAND", "KM N"]': 14},
 {'["CABARRUS", "02-01"]': 12},
 {'["WAKE", "12-04"]': 2},
 {'["ORANGE", "CH"]': 11},
 {'["ORANGE", "KM"]': 11},
 {'["MECKLENBURG", "074"]': 13},
 {'["RANDOLPH", "DR"]': 11},
 {'["WAKE", "07-12"]': 2},
 {'["ORANGE", "CP"]': 11},
 {'["MECKLENBURG", "144"]': 13},
 {'["MECKLENBURG", "083"]': 13},
 {'["MONTGOMERY", "T2"]': 11},
 {'["WAKE", "01-43"]': 9},
 {'["MECKLENBURG", "243"]': 13}]

## Phase 1 takeaway

This notebook validates the Phase 1 ingestion workflow for the current Atlas inputs.

Main findings:
- The non-county baseline atlas aligns cleanly with `NC_pct21.json` at the precinct-unit level.

- The county-preserving atlas does not appear to use the same unit representation.

- In the current first pass, the county-preserving run yields 700 assignment units, while `NC_pct21.json` contains 2,650 precinct-style graph units.

- This suggests the county-preserving atlas is using a different geography or multilevel graph representation and will require the exact matching graph/geography file or a crosswalk before direct precinct-level comparison.

Immediate implication:

- The non-county baseline is ready for the next phase of analysis.

- The county-preserving comparison requires additional reconciliation before it can be used in the same precinct-level workflow.